# मानव-इन-द-लूप: पूर्व-कारबाही गेटहरू, जोखिम तह निर्धारण, र अडिट लगिङ

यसपाठको README ले मानव-इन-द-लूपलाई छोटो स्निपेटबाट परिचय गराउँछ जसले प्रयोगकर्तालाई एजेन्टले प्रतिक्रिया उत्पादन गरेपछि `APPROVE` वा `REJECT` सोध्छ। त्यो ढाँचा राम्रो सुरुवात बिन्दु हो, तर उत्पादन HITL कार्यान्वयनहरूले सामान्यतया तीन थप भागहरू चाहिन्छन्:

1. एउटा **पूर्व-कारबाही गेट** जुन एजेन्टले जोखिमपूर्ण कदम चाल्नु **अघि** चल्छ, जसले लागत, अपरिवर्तनीयता, र ढिलाइ नियन्त्रणमा राख्न मद्दत गर्छ।
2. **जोखिम तह निर्धारण**, जसले कम जोखिम कार्यहरूलाई स्वतः कार्यान्वयन गर्न, मध्यम जोखिम कार्यहरूलाई ब्याच-अनुमोदन गर्न, र उच्च जोखिम कार्यहरूलाई मात्र मानवमा अवरोध गर्न अनुमति दिन्छ।
3. एउटा **अडिट लग र संशोधन लूप**, जसमा प्रत्येक गेट निर्णय JSONL रूपमा रेकर्ड हुन्छ, र अस्वीकारले एजेन्टलाई संरचित कारणसहित पुनःप्रेरित गर्दछ, केवल `Revising...` प्रिन्ट गर्नुको सट्टा।

यो नोटबुकले यी प्रत्येकलाई `06-system-message-framework.ipynb` जस्तै प्राथमिक तत्वहरूमा बनाउँछ। यो `DEMO_MODE = True` (इन्टरएक्टिभ इनपुट आवश्यक छैन) मा वा वास्तविक `input()` प्राँप्टहरुमा `DEMO_MODE = False` हुँदा अन्त्य-देखि-अन्त्य चल्छ। नोट: DEMO_MODE मा तेस्रो लक्ष्यको पुन: प्रयास स्क्रिप्ट गरिएको छ जसले लूपको क्रियाविधि अन्त्य-देखि-अन्त्य देखिन्छ। वास्तविक संशोधन-चालित पुनर्वर्गीकरणको लागि `DEMO_MODE = False` र एउटा अपरेटर आवश्यक छ।

**क्षेत्र बाहिर (अर्का पाठहरूमा सम्हालिएको):** प्रमाणीकरण र पहुँच नियन्त्रण (पाठ 06 README खतरा 2), उपकरण-कल मिडलवेयर (पाठ 14 MAF गहिरो विश्लेषण), बहु-एजेन्ट बहस ढाँचाहरू।


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## ढाँचा १: पूर्व-ककार्य गेट

README को HITL स्निपेटले पहिले एजेन्टलाई कल गर्छ, त्यसपछि प्रयोगकर्तालाई आउटपुट अनुमोदन गर्न सोध्छ। त्यो एक **पोस्ट-ककार्य** प्रवाह हो। एजेन्टले पहिले नै कार्य गरिसकेको हुन्छ, त्यसैले LLM कल लागत पहिले नै तिर्नु भएको हुन्छ, र कुनै पनि प्रतिक्रिया (इमेल पठाइएको, डाटाबेस पंक्ति लेखिएको, टिप्पणी पोस्ट गरिएको) पहिले नै भइसकेको हुन्छ।

एक **पूर्व-ककार्य** प्रवाहले एजेन्टले जोखिमपूर्ण कदम चाल्नु अघि गेट राख्छ। एजेन्टले कार्य प्रस्ताव गर्दछ, गेटले कार्यान्वयन गर्ने कि नगर्ने निर्णय गर्छ, र मात्र अनुमोदन पछि प्रतिक्रिया हुन्छ।

| पक्ष | पोस्ट-ककार्य अनुमोदन (README स्निपेट) | पूर्व-ककार्य गेट (यो नोटबुक) |
|---|---|---|
| अनुमोदन कहिले चल्छ? | एजेन्टले आउटपुट उत्पादन गरेपछि | कुनै पनि प्रतिक्रिया हुनु अघि |
| अस्वीकृतिमा LLM लागत | पहिले नै तिर्नु भएको | प्रस्ताव मात्रको लागि तिर्नु, कार्यको लागि होइन |
| अपरिवर्तनीय प्रतिक्रिया | सम्भव (कार्य पहिले नै भइसकेको) | रोकिएको |
| अडिट स्पष्टता | अनुमोदन एक प्रिन्ट स्टेटमेन्ट हो | अनुमोदन टाइमस्ट्याम्प, कार्य, कारणसहित JSON रेकर्ड हो |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## ढाँचा २: जोखिम स्तर निर्धारण

हरेक कार्यलाई मानवीय अनुमोदन आवश्यक पर्दैन। सार्वजनिक API विरुद्धको मात्र पढ्ने खोजीले ग्राहकलाई इमेल पठाउनुभन्दा फरक जोखिमहरू राख्दछ। दुबैलाई एउटै तरिकाले ह्यान्डल गर्दा अपरेटरको ध्यान बर्बाद हुन्छ र एजेन्ट सुस्त हुन्छ।

एउटा सरल ३-स्तरीय मोडल:

| स्तर | उदाहरणहरू | अनुमोदन प्रक्रिया |
|---|---|---|
| `कम` (केवल पढ्न) | ज्ञानको आधार खोज्ने, फ्लाइट विकल्पहरू खोज्ने, सार्वजनिक वेब पृष्ठ ल्याउने | स्वतः कार्यान्वयन, लेखा परीक्षणको लागि लग गरिन्छ |
| `मध्यम` (सस्तो परिवर्तन) | परिणाम क्याच गर्ने, काउन्टर बढाउने, स्मरणपत्र तालिका बनाउने | स्वतः कार्यान्वयन तर दैनिक समिक्षाको लागि ब्याच गरिन्छ |
| `उच्च` (बाह्य-सामना गर्ने वा अपरिवर्तनीय) | इमेल पठाउने, कार्डबाट चार्ज गर्ने, सार्वजनिक च्यानलमा पोष्ट गर्ने | मानवीय अनुमोदनमा रोक्नुहोस् |

यो एउटा स्तर निर्धारण हो। उत्पादन प्रणालीहरूले प्रायः थप सूक्ष्म स्तरहरू प्रयोग गर्छन् (जस्तै, AWS IAM अनुमति स्तरहरू, भूमिका-आधारित पहुँच स्तरहरू)। तलको ३-स्तरीय संस्करण एउटा एजेन्टका लागि सबभन्दा सानो उपयोगी संस्करण हो जसले मात्र पढ्ने र साइड-इफेक्ट गर्ने कार्यहरू मिश्रित गर्छ।

तल दिइएको वर्गीकरणकर्ता शब्द कुञ्जी अड्चनहरू प्रयोग गर्छ ताकि डेमो स्थिर र सस्तो रहोस्। उत्पादन प्रणालीमा तपाईंले यसलाई सिकाइएको वर्गीकरणकर्ता वा नीति इन्जिनसँग प्रतिस्थापन गर्नुहुनेछ।


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## ढाँचा 3: अडिट लग र संशोधन लूप

एक `print("Response approved.")` अडिट लग होइन। विश्वासका लागि, प्रत्येक गेट निर्णय संरचित घटनाको रूपमा रेकर्ड गर्नुपर्छ जसलाई पछि तपाईं सोध्न, पुन: प्रर्दशन गर्न, वा घटनाको समीक्षा संग जोड्न सक्नुहुन्छ।

दुई भागहरू:

1. **Append-only JSONL।** प्रत्येक निर्णयको लागि एक लाइन, जसमा टाइमस्ट्याम्प, क्रिया, तह, निर्णय, कारण समावेश हुन्छ। grep गर्न सजिलो, पछि वास्तविक लग स्टोरमा पठाउन सजिलो।
2. **अस्वीकृतिमा संशोधन लूप।** जब गेटले `deny` फर्काउँछ, एजेन्टले अस्वीकृतिको कारण सन्दर्भमा राखेर आफैंलाई पुन: प्रॉम्प्ट गर्छ, ताकि अर्को प्रस्तावले समस्याबाट बच्न सकोस्।


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## थप स्रोतहरू

यी HITL ढाँचाहरूका विभिन्न भेरियन्टहरू कार्यान्वयन गर्ने अन्य धेरै सार्वजनिक परियोजनाहरू छन्। के तपाईँको स्ट्याकसँग के उपयुक्त हुन्छ भनेर थाहा पाउन दृष्टिकोणहरू तुलना गर्नुहोस्:

- **LangChain** मानव-इन-द-लूप उपकरण रैपरहरू ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): मानव इनपुटको लागि कार्यान्वयन रोक्ने ड्रप-इन उपकरण रैपरहरू।
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ले यसलाई पुनर्संरचना गरेको छ): बहु-एजेन्ट संवादहरूमा मान्छेलाई प्रतिनिधित्व गर्न विशेष रूपमा एजेन्ट भूमिका प्रयोग गर्दछ।
- **Microsoft Agent Framework (MAF)** function-invocation मिडलवेयर ([docs](https://learn.microsoft.com/agent-framework/)): प्रत्येक उपकरण/फंक्शन कलको वरिपरि चल्ने मिडलवेयर, गेटिंग लॉजिक र अनुमोदन प्रवाहका लागि उपयुक्त।

प्रत्येक परियोजनाले यी तीन उप-ढाँचाहरूलाई फरक फरक तरिकाले सञ्चालित गर्दछ: LangChain ले तिनीहरूलाई उपकरणको रूपमा र्याप गर्दछ, AutoGen ले एजेन्ट भूमिका प्रयोग गर्दछ, र Microsoft Agent Framework ले function-invocation मिडलवेयर प्रयोग गर्दछ। आफ्नै एजेन्टको डिजाइन छान्नुभन्दा पहिले एक वा दुई कार्यान्वयनहरू पूर्ण रूपमा पढ्नुहोस्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
